# 🐍 Clase 11 · Primera estrategia + métricas

> Someter el motor completo de L1–L10 a una estrategia con señal operativa y métricas honestas — PnL, posición, parent arrival para la decisión y decision mid para la ejecución de cada orden hija. Checkpoint integrador que depende directamente del contrato y el runner de L10.

**Hoy construyes:** medir una estrategia contra un benchmark.

⏱️ 🟢 LIVE ~20 min.

### Cómo funciona este cuaderno

Cada ejercicio declara una ruta pedagógica, decidida por contenido y no por posición: **🟢 LIVE** (núcleo presencial) · **🔵 REQUIRED** (consolidación autónoma requerida y evaluable) · **🟣 OPTIONAL** (profundización no obligatoria y no evaluable). Escribe tu respuesta, ejecuta la **✅ comprobación plegada** con `Shift+Enter` y usa la pista o solución solo cuando la necesites.

### 1. Estrategia de imbalance

<sub>🟢 LIVE · núcleo presencial · ~5 min</sub>

Define `ImbalanceStrategy(thr)`: compra 0.05 si `imbalance(3) > thr`, vende 0.05 si `< -thr`. Market orders.

<sub>practicas: señal larga/corta</sub>

In [ ]:
from exchange import Strategy, NewOrder, Order, Side, OrderType, Market, Backtest
class ImbalanceStrategy(Strategy):
    def __init__(self, thr=0.3):
        self.thr = thr
    def on_book_update(self, book):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert ImbalanceStrategy.on_book_update.__code__.co_consts != (None,), '⏸ implementa ImbalanceStrategy.on_book_update: su cuerpo sigue siendo pass'
r = Backtest(Market.sample(), ImbalanceStrategy(0.3)).run()
assert r.n_steps == 500
print('ok ', r)

<details>
<summary>💡 Ver solución</summary>

```python
class ImbalanceStrategy(Strategy):
    def __init__(self, thr=0.3):
        self.thr = thr
    def on_book_update(self, book):
        imb = book.imbalance(3)
        if imb is None: return []
        if imb > self.thr:
            return [NewOrder(Order('BTCUSDT', Side.BUY, 0.05, order_type=OrderType.MARKET))]
        if imb < -self.thr:
            return [NewOrder(Order('BTCUSDT', Side.SELL, 0.05, order_type=OrderType.MARKET))]
        return []
```

</details>

### 2. Mídela

<sub>🟢 LIVE · núcleo presencial · ~5 min</sub>

Corre la estrategia y guarda `equity`, `pos` y `fills` del resultado.

<sub>practicas: leer BacktestResult</sub>

In [ ]:
from exchange import Strategy, NewOrder, Order, Side, OrderType, Market, Backtest
class ImbalanceStrategy(Strategy):
    def __init__(self, thr=0.3): self.thr=thr
    def on_book_update(self, book):
        imb = book.imbalance(3)
        if imb is None: return []
        if imb > self.thr: return [NewOrder(Order('BTCUSDT', Side.BUY, 0.05, order_type=OrderType.MARKET))]
        if imb < -self.thr: return [NewOrder(Order('BTCUSDT', Side.SELL, 0.05, order_type=OrderType.MARKET))]
        return []
equity = None
pos = None
fills = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert equity is not None, '⏸ equity sigue en None: completa el ejercicio antes de validar'
assert pos is not None, '⏸ pos sigue en None: completa el ejercicio antes de validar'
assert fills is not None, '⏸ fills sigue en None: completa el ejercicio antes de validar'
assert isinstance(equity, float) and isinstance(fills, int)
print('ok  equity=%.2f pos=%.3f fills=%d' % (equity, pos, fills))

<details>
<summary>💡 Ver solución</summary>

```python
r = Backtest(Market.sample(), ImbalanceStrategy(0.3)).run()
equity = r.final_equity
pos = r.final_position
fills = r.n_fills
```

</details>

### 3. Parent arrival

<sub>🟢 LIVE · núcleo presencial · ~5 min</sub>

Guarda `arrival_mid`, el parent-order arrival: mid del primer snapshot del mercado.

<sub>practicas: primer mid de la decisión completa</sub>

In [ ]:
from exchange import Market
arrival_mid = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert arrival_mid is not None, '⏸ arrival_mid sigue en None: completa el ejercicio antes de validar'
assert 90000 < arrival_mid < 110000
print('ok  arrival_mid=%.2f' % arrival_mid)

<details>
<summary>💭 Pista (antes de mirar la solución)</summary>

Encadena: `Market.sample()` te da el mercado, `.step()` el primer libro, `.mid` su punto medio.

</details>

<details>
<summary>💡 Ver solución</summary>

```python
from exchange import Market
arrival_mid = Market.sample().step().mid
```

</details>

### 4. Riesgo escondido

<sub>🟢 LIVE · núcleo presencial · ~5 min</sub>

Para `thr=0.1` y `thr=0.5`, guarda en `pos_small_thr` y `pos_big_thr` la posición final. Un umbral bajo opera más y arriesga más inventario.

<sub>practicas: interpretar inventario</sub>

In [ ]:
from exchange import Strategy, NewOrder, Order, Side, OrderType, Market, Backtest
class ImbalanceStrategy(Strategy):
    def __init__(self, thr=0.3): self.thr=thr
    def on_book_update(self, book):
        imb = book.imbalance(3)
        if imb is None: return []
        if imb > self.thr: return [NewOrder(Order('BTCUSDT', Side.BUY, 0.05, order_type=OrderType.MARKET))]
        if imb < -self.thr: return [NewOrder(Order('BTCUSDT', Side.SELL, 0.05, order_type=OrderType.MARKET))]
        return []
pos_small_thr = None
pos_big_thr = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert pos_small_thr is not None and pos_big_thr is not None
print('ok  thr0.1 pos=%.3f | thr0.5 pos=%.3f' % (pos_small_thr, pos_big_thr))

<details>
<summary>💭 Pista (antes de mirar la solución)</summary>

Dos backtests idénticos salvo el umbral: `Backtest(Market.sample(), ImbalanceStrategy(0.1)).run().final_position`, y lo mismo con 0.5. `Market.sample()` nuevo en cada uno, para que ambos vean el mismo día.

</details>

<details>
<summary>💡 Ver solución</summary>

```python
pos_small_thr = Backtest(Market.sample(), ImbalanceStrategy(0.1)).run().final_position
pos_big_thr = Backtest(Market.sample(), ImbalanceStrategy(0.5)).run().final_position
```

</details>

## Cierre

Sin benchmark no hay estrategia: parent arrival juzga la decisión; cada decision mid juzga su ejecución.

Cada ejercicio lleva una ruta explícita: LIVE, REQUIRED u OPTIONAL. Sigue REQUIRED para el itinerario autónomo y elige OPTIONAL solo si tienes margen.

**L12:** seguimos construyendo el sistema sobre esta pieza.

## 🚀 Llévatelo a un `.py`

Un notebook va genial para explorar, pero el código de verdad vive en archivos `.py` que se ejecutan enteros de una vez. Abre **`judge.py`**: es lo que acabas de construir, ordenado y de una pieza.

Ejecútalo desde una terminal:

```bash
python judge.py
```

…o aquí mismo, en la siguiente celda:

In [ ]:
!python judge.py

> Es la misma pieza que vive en el paquete `exchange/` — aquí, condensada en un archivo que puedes leer de una sentada.